## 语言模型实战
### 在本节中，我们将加载一个现成的小型预训练Transformer模型并与之交互，从宏观层面理解这类模型的工作原理。近年来，各大企业、研究实验室以及开源社区已经发布了数千款可供使用的开源模型。我们会选用一款可以直接在本地硬件上运行的小型模型，但要注意，后续推出的那些规模更大（体量超其百倍）、能力更强的模型，遵循的原理与之完全相同。下面是几款不错的小型模型示例：

+ GPT-2（1.37亿参数）该模型在2019年凭借当时亮眼的文本生成能力引发广泛关注。以如今的标准来看，它的规模很小，甚至略显朴素，但GPT-2能够很好地展示这类语言模型的运行机制。
+ Qwen2（4.94亿参数）这是阿里推出的模型，属于通义千问（Qwen）系列。通义千问系列包含参数规模从5亿到超1000亿的多款模型。
+ SmolLM（1.35亿参数）这是Hugging Face推出的模型，使用极高质量的数据完成训练。作者发布了1.35亿、3.6亿以及17亿参数版本的模型。

## Tokenizing Text（文本分词）
+ 让我们开启基于初始输入生成文本的学习之旅。例如，给定短语“it was a dark and stormy”，我们希望模型生成词语来续写这句话。模型无法直接接收文本作为输入；它们的输入必须是以数字形式表示的数据。要将文本输入模型，我们首先需要找到一种方法，把文本序列转换为数字。这个过程叫做分词，是所有自然语言处理流程中至关重要的一步。

In [6]:
from transformers import AutoTokenizer

In [ ]:
# model_dir = "Qwen/Qwen2.5-7B-Instruct"
model_dir = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
prompt = "四月的江南，湖面上"
inputs = tokenizer(prompt, return_tensors="pt", return_attention_mask=True)
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]
print(input_ids)

tensor([[ 63703,   9754,   9370, 105811,   3837,  99529, 101653]])


In [8]:
for t in input_ids:
    print(t, "\t:", tokenizer.decode(t))

tensor([ 63703,   9754,   9370, 105811,   3837,  99529, 101653]) 	: 四月的江南，湖面上


In [ ]:
prompt = "It was a dark and stormy"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
print(input_ids)

+ 分词器会将输入字符串拆分为一系列词元，并为每个词元分配唯一ID。大多数单词由单个词元表示，但在使用通义千问（Qwen）和GPT-2分词器时，“stormy”会被拆成两个词元：一个代表“ storm”（包含单词前面的空格），另一个代表后缀“y”。
+ 这样模型就能学习到“stormy”与“storm”存在关联，并且后缀“y”常用来将名词转化为形容词。与之不同的是，SmolLM分词器不会对该特定句子中的任何单词进行拆分。
+ 每个模型通常配套专属的分词器，因此使用模型时务必选用对应的分词器。本节提到的这三个模型，词表规模在50000至150000个词元之间，足以对几乎所有输入文本进行编码表示。

+ 虽然我们经常会提到训练分词器，但这和模型训练完全不是一回事。模型训练本身具有随机性（非确定性），而我们训练分词器采用的是统计方法，该方法会识别出针对给定数据集，哪些子词是最优选择。如何挑选子词属于分词算法的设计决策。因此，分词器训练是确定性的。我们不深入探讨各类分词策略，不过目前最主流的一些子词算法包括：GPT-2所使用的字节级字节对编码（BPE）、词块（WordPiece）以及句子片段（SentencePiece）。

In [9]:
for t in input_ids:
    print(t, "\t:", tokenizer.decode(t))

tensor([ 63703,   9754,   9370, 105811,   3837,  99529, 101653]) 	: 四月的江南，湖面上


In [10]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_dir)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## Predicting Probabilities（预测概率）

+ GPT-2、通义千问（Qwen）与SmolLM均作为因果语言模型（也叫自回归模型）进行训练，这意味着它们的训练目标是：根据序列中前面的词元，预测序列的下一个词元。
+ 如果我们将上一节中经过分词处理的句子输入模型，得到的结果会为输入字符串里的每个分词生成151936个数值：

In [11]:
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
outputs = model(input_ids)
outputs.logits.shape  # An output for each input token

torch.Size([1, 7, 151936])

+ 输出的第一个维度是批次数量（值为1，因为我们仅向模型传入了单个序列）。第二个维度是序列长度，也就是输入序列中的词元数量（本例中为7）。第三个维度是词表大小。对于原始序列里的每一个词元，我们都会得到一组约150000个数值。这些就是模型原始输出，即对数几率，对应词表中的各个词元。对于每一个输入词元，模型都会预测词表中每个词元在该位置接续序列的概率。

In [12]:
final_logits = model(input_ids).logits[0, -1]  # The last set of logits
final_logits.argmax()  # The position of the maximum

tensor(99804)

+ 我们可以使用argmax()方法找到数值最大的token对应的索引

In [ ]:
tokenizer.decode(final_logits.argmax())

+ 现在我们通过topk()选取前10个值，找出其他哪些token曾是候选对象。

In [13]:
import torch

top10_logits = torch.topk(final_logits, 10)
for index in top10_logits.indices:
    print(tokenizer.decode(index))

波
的
，
泛
飘
水
风
碧
白
荡


+ 我们需要将对数几率转换为概率，以便更好地理解模型对每个预测结果的置信度。实现方式是将每个数值与其余所有预测值进行对比并做归一化处理，使所有数值之和等于1。这正是softmax()运算的作用。下面这段代码使用softmax()输出模型预测中概率最高的前10个词元及其对应的概率：

In [ ]:
top10 = torch.topk(final_logits.softmax(dim=0), 10)
for value, index in zip(top10.values, top10.indices):
    print(f"{tokenizer.decode(index):<10} {value.item():.2%}")

## Generating Text （生成文本）
+ 一旦我们掌握获取模型对序列中下一个token的预测结果的方法，只需反复将模型的预测结果回传给模型自身，就能轻松生成文本。我们可以调用model(ids)，生成一个新的token编号，将其加入序列列表，再重新调用该函数。为了更便捷地生成多个词语，transformers库中的自回归模型提供了适用于该场景的generate()方法。

In [14]:
output_ids = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=20)  # type: ignore
decoded_text = tokenizer.decode(output_ids[0])
print("Input IDs", input_ids[0])
print("Output IDs", output_ids)
print(f"Generated text: {decoded_text}")

Input IDs tensor([ 63703,   9754,   9370, 105811,   3837,  99529, 101653])
Output IDs tensor([[ 63703,   9754,   9370, 105811,   3837,  99529, 101653, 100461,  71618,
         109188, 120497, 121508,   1773,  99529,  52510, 113159,  29524, 100811,
           3837, 100655,  99261,  18493, 107629, 101972, 109613,  29490,  82894]])
Generated text: 四月的江南，湖面上泛起层层涟漪。湖水清澈如镜，鱼儿在水中自由自在地游


![](../pictures/gtext01.png)

In [ ]:
beam_output = model.generate(input_ids, num_beams=5, max_new_tokens=30)  # type: ignore
print(tokenizer.decode(beam_output[0]))

+ 束搜索这类技术不会逐个选择一个标记，而是会探索该序列的多种可能后续片段，并返回概率最高的后续序列。该方法在生成过程中保留概率最高的num_beams个候选假设，并从中选出概率最大的那一个。

+ repetition_penalty
+ 对已生成的token施加多大惩罚，以此避免重复。合适的默认值为1.2。

In [ ]:
beam_output = model.generate(  # type: ignore
    input_ids, num_beams=5, repetition_penalty=2.0, max_new_tokens=38
)
print(tokenizer.decode(beam_output[0]))

+ 束搜索的速度比贪心搜索更慢，因为它需要同时对多条候选束执行推理，对于大模型而言这会成为一个难题。
+ 当我们使用贪心搜索和束搜索做生成时，会迫使模型按照高概率下一词的分布生成文本。有意思的是，高质量的人类语言并不遵循类似的分布。人类写出的文本往往更难预测。一篇优秀论文的作者针对这一反直觉现象提出猜想：人类语言不倾向使用可预测的词汇，人们会刻意避免陈述显而易见的内容。
+ 该论文提出了一种名为核采样的方法。在介绍核采样之前，我们先谈谈采样。采样机制会从下一个词元的概率分布中抽样，选出下一个词元。这意味着采样并非确定性的生成过程。假设候选下一词元为“夜晚”（60%）、“白天”（35%）和“苹果”（5%），此时不会像贪心搜索那样直接选择“夜晚”，而是从这个分布里做抽样。也就是说，即便是低概率的词元“苹果”，也有5%的概率被选中，可能生成无意义的文本。采样可以避免生成重复文本，因此能够产出更多样化的内容。

+ 模型中通过使用do_sample参数来执行采样操作

In [ ]:
sampling_output = model.generate(input_ids, do_sample=True, max_new_tokens=34, top_k=0)  # type: ignore
print(tokenizer.decode(sampling_output[0]))

## temperature （温度）
+ 我们可以在概率分布采样之前对其进行调整，借助温度参数让概率分布变得更尖锐或更平缓。温度值大于1时，会提升分布的随机性，可用于生成概率较低的词元。温度取值在0到1之间时，会降低随机性，提高高概率词元的出现概率，避免生成过于出人意料的预测结果。温度设为0时，全部概率会集中到最可能出现的下一个词元上，这等价于贪心解码。

In [ ]:
sampling_output = model.generate(  # type: ignore
    input_ids, do_sample=True, temperature=0.4, max_new_tokens=40, top_k=0
)
print(tokenizer.decode(sampling_output[0]))

In [ ]:
sampling_output = model.generate(  # type: ignore
    input_ids,
    do_sample=True,
    temperature=0.001,
    max_new_tokens=40,
    top_k=0,
)
print(tokenizer.decode(sampling_output[0]))

In [ ]:
sampling_output = model.generate(  # type: ignore
    input_ids,
    do_sample=True,
    temperature=3.0,
    max_new_tokens=40,
    top_k=0,
)
print(tokenizer.decode(sampling_output[0]))

In [ ]:
sampling_output = model.generate(input_ids, do_sample=True, max_new_tokens=40, top_k=5)  # type: ignore
print(tokenizer.decode(sampling_output[0]))

In [ ]:
sampling_output = model.generate(  # type: ignore
    input_ids, do_sample=True, max_new_tokens=40, top_p=0.94, top_k=0
)
print(tokenizer.decode(sampling_output[0]))

#### top_k and top_p

>top_k指的是在每一步生成时，仅从概率最高的k个词中进行采样。也就是说，模型会把词汇表中概率排名在前k的词筛选出来，而忽略其余词。这有助于避免生成低概率、可能不合理的词，从而提升生成文本的质量。不过，若k值设置得太小，生成的文本可能会缺乏多样性；若设置得太大，则可能会引入一些不合理的词。

>top_p（也被称作核采样）指的是在每一步生成时，仅从累积概率超过p的最小词集里进行采样。模型会按照概率对词汇表中的词进行排序，接着选取累积概率大于等于p的最小词集，最后从这个词集中采样。这种方法能够自适应地调整采样的词集大小，在保证生成文本质量的同时，提升文本的多样性。

#### top_k和top_p一起使用的含义

>当同时使用top_k和top_p时，模型会先运用top_k筛选出概率最高的k个词，然后在这k个词里，再依据top_p的规则，选取累积概率超过p的最小词集，最后从这个最终的词集中进行采样。这种组合方式能够综合二者的优势，既避免生成低概率的词，又保证生成文本具有一定的多样性。

### 累积概率的定义
在文本生成中，模型会为词汇表中的每个词输出一个概率，表示该词作为下一个生成词的可能性。当我们按照概率对这些词进行降序排序后，累积概率就是从概率最高的词开始，逐个累加每个词的概率所得到的结果。

### 累积概率的计算示例
假设词汇表中有5个词，模型输出的这些词的概率分别如下：

| 词 | 概率 |
| --- | --- |
| 词A | 0.4 |
| 词B | 0.3 |
| 词C | 0.2 |
| 词D | 0.08 |
| 词E | 0.02 |

我们按照概率从高到低对这些词进行排序，然后计算累积概率：

| 词 | 概率 | 累积概率 |
| --- | --- | --- |
| 词A | 0.4 | 0.4 |
| 词B | 0.3 | 0.4 + 0.3 = 0.7 |
| 词C | 0.2 | 0.7 + 0.2 = 0.9 |
| 词D | 0.08 | 0.9 + 0.08 = 0.98 |
| 词E | 0.02 | 0.98 + 0.02 = 1.0 |

### 在`top_p`采样中使用累积概率
当我们设置`top_p`参数时，比如`top_p = 0.9`，模型会从概率最高的词开始，依次累加词的概率，直到累积概率超过或等于`p`（这里是0.9）。在上述示例中，累积概率超过0.9的最小词集包含词A、词B和词C，因为到词C时累积概率达到了0.9。所以，模型会从词A、词B和词C中进行采样来生成下一个词。

### 结合`top_k`和`top_p`时的累积概率计算
当同时使用`top_k`和`top_p`时，模型会先使用`top_k`筛选出概率最高的`k`个词，然后在这`k`个词中计算累积概率，并根据`top_p`的规则选取最终的采样词集。

例如，假设`top_k = 3`，那么模型会先筛选出词A、词B和词C。然后在这3个词中计算累积概率：

| 词 | 概率 | 累积概率 |
| --- | --- | --- |
| 词A | 0.4 | 0.4 |
| 词B | 0.3 | 0.4 + 0.3 = 0.7 |
| 词C | 0.2 | 0.7 + 0.2 = 0.9 |

如果`top_p = 0.9`，由于在这`k`个词中累积概率达到0.9时包含了所有3个词，所以模型会从词A、词B和词C中进行采样。

通过这种方式，累积概率帮助模型在采样过程中动态地确定合适的词集，从而平衡生成文本的质量和多样性。 